In [1]:
# UOV PROTOCOL
def KeyGen(n, m, o, q):
    '''
    returns a tuple (
                        pk: List of coefficient matrics,
                        sk: (List of coefficient matrices, Matrix
                     )
    containing the public key and the secret key.
    The secret is again a tuple (FF, M), where
            - FF is the central map and
            - M an invertible linear transformation.
    '''
    F = GF(q)
    FF = []
    # pick the quadratic equations at random
    for i in range(m):
        FF.append(generate_f(n, o, q))
    # print("FF[2] \n", FF[2])
    # choose an invertible linear transform F_n -> F_n
    M = GL(n, F).random_element().matrix()
    sk = (FF, M)
    # construct the public key
    PP = []
    for i in range(len(FF)):
        PP.append(M.transpose()*FF[i]*M)
    # print("PP[2] \n", PP[2])
    pk = PP
    # print(type(pk), type(sk))
    return pk, sk

In [2]:
def sign_uov(n, m, o, q, sk, t):

    '''
    returns the UOV signature as a vector of length n
    '''
    assert (len(t) == m)
    F = GF(q)
    R = PolynomialRing(F, n, 'x')
    x = vector(R.gens())

    # we skip the hashing part
    i = 1
    while true:
        try:
            sol_temp = solve_FF(n, m, o, q, sk, t, F, R, x)
            # print("sol", sol_temp)
            break
        except ValueError:
            i+=1
            # system has no solutions
            print(f'Hélas ... Too much vinegar. Poging {i}')
    # print("de inverse transformatie", sk[1].inverse())
    # print('element', sol_temp[2])
    # create s'
    sol = [sol_temp[i] for i in range(o)]
    sol.extend([x[i] for i in range(o, n)])
    assert (len(sol) == n)
    # print(sol)
    signature = sk[1].inverse()*vector(sol)
    return signature

def solve_FF(n, m, o, q, sk, t, F, R, x):
    # pick random values for the vinegar variables
    for i in range(n-o):
        # print(i+o-1)
        x[i+o] = F.random_element()
    # print(x)
    # solve the system F(s') = t (m linear equations in o variables, m <= o)
    FF, M = sk
    system = [x*FF[i]*x - t[i] for i in range(m)]
    A = matrix([ [eq.coefficient(v) for v in x[0:o]] for eq in system])
    # TODO try with and without max
    b = vector([-eq.constant_coefficient() for eq in system])
    # print("het rechterlid", b)
    sol = A.solve_right(b)
    return sol

In [3]:
def verify(n, m, o, q, PP, signature):
    F = GF(q)
    R = PolynomialRing(F, n, 'x')
    x = vector(R.gens())
    t_bar = []
    for i in range(len(PP)):
        t_bar.append(signature*PP[i]*signature)
    return t_bar

In [4]:
def generate_f(n, o, q):
    '''
    generates a random quadratic equation over F_q in n variables.
    q has to be odd.
    '''
    F = GF(q)
    # define the polynomial ring where the variables live
    R = PolynomialRing(F, n, 't', order='deglex')
    # construct a quadratic polynomial over F_q, such that every term has a vinegar variable
    f_k = sum(R.gens()[i]*sum(F.random_element()*R.gens()[j] for j in range(n)) for i in range(o, n)) 
    # print(type(f_k))
    # print(f_k.coefficients())
    size = len(R.gens())
    gram = Matrix(F, size, size)
    for i in range(size):
        quadratic_terms = {R.gens()[i]: 2}
        gram[i, i] = f_k.coefficient(quadratic_terms)
        for j in range(i+1, n):
            other_terms = {R.gens()[i]: 1, R.gens()[j]: 1}
            gram[i, j] = f_k.coefficient(other_terms)
    

    # Q = QuadraticForm(f_k);
    # # gets matrix of polar form
    # gram = Q.matrix() * F(2)**(-1);
    
    # further checks
    x = vector(R.gens())
    assert (x * gram * x == f_k)
    return gram

In [5]:
# TESTSETUP
def generate_UOV_variables(n, m, o, q):
    v = n - o
    F = GF(q)
    pk, sk = KeyGen(n, m, o, q)
    FF, T = sk
    P = PolynomialRing(F, n, 't', order='deglex')
    gens_vector = vector(P.gens())
    oil = gens_vector[0:o]
    vinegar = gens_vector[o:]
    # print('oil and vinegar \t', oil, vinegar)
    scrabbled = T*gens_vector
    # print('scrabbled variables \t',scrabbled)

    O_bar = block_matrix([
        [identity_matrix(F, o), zero_matrix(F, o, n-o)]
        ])

    O = (T.inverse()*O_bar.transpose()).transpose()
    V_bar = block_matrix([
        [zero_matrix(F, v, o), identity_matrix(F, v)]
        ])
    V = (T.inverse()*V_bar.transpose()).transpose()
    # print('O \n', O)
    return pk, sk, P, oil, vinegar, O, V, gens_vector

In [6]:
def create_col_index(x, n, d, col_temp=None, degree_index=0):
    if col_temp is None:
        col_temp = 1
    if n == d:
        res = 1
        for el in x:
            res *= el
        return res
    col_index = []
    
    # base case
    if len(list(col_temp.factor())) == d:
        # print(col_temp)
        return [col_temp]
        
    # extend the solutions
    ext_sol = []
    for extension_index in range(degree_index, min(n - d + degree_index + 1, n)): # TODO understand the index issue
        temp_sol = col_temp*x[extension_index]
        ext_sol.append((temp_sol, extension_index))
        
    # call creat_col_index recursively
    for sol in ext_sol:
        res = create_col_index(x, n, d, sol[0], sol[1]+1)
        # print(res)
        if [res] != []:
            col_index += res
        # print(possible_start_index, col_index)
    return col_index

n, m, o, q = 5, 2, 2, 2
F = GF(q)
d = o
R = PolynomialRing(F, n, 'x')
# print(R.monomials_of_degree(2))
print(create_col_index(R.gens(), n, d))

[x0*x1, x0*x2, x0*x3, x0*x4, x1*x2, x1*x3, x1*x4, x2*x3, x2*x4, x3*x4]


In [7]:
# EXPERIMENTS
def experiment_1_sampling_vectors(n, m, o, q):
    pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
    F = GF(q)
    v = n - o
    # EXPERIMENT 1
    test_it = q^n
    counter_O, counter_V, counter_random = 0, 0, 0
    counter_V2 = 0
    counter_random_2 = 0

    # convert the public key to polynomials
    pk_poly = []
    for coef_matrix in pk:
        polynomial = P(gens_vector*coef_matrix*gens_vector)
        pk_poly.append(polynomial)
    for i in range(test_it):
        random_O_vector = vector([F.random_element() for i in range(o)])*O
        random_V_vector = vector([F.random_element() for i in range(v)])*V
        random_vector = vector([F.random_element() for i in range(n)])
        if all((poly(list(random_O_vector)) == 0) for poly in pk_poly):
            counter_O += 1
        if all((poly(list(random_V_vector)) == 0) for poly in pk_poly):
            counter_V += 1
        if all((poly(list(random_vector)) == 0) for poly in pk_poly):
            counter_random += 1
    res_O = round(counter_O/test_it, 2)
    res_V = round(counter_V/test_it, 2)
    res_both = round(counter_random/test_it, 2)
    # print('oil vectors  \t \t', res_O)
    # print('vinegar vectors \t', res_V, "\t expected: \t", round(q**(-o), 2))
    # print('random vectors \t \t', res_both, "\t expected: \t")
    return res_O, res_V, res_both

    # print('random O vector \t', random_O_vector, 'evaluation \t',  p1(list(random_O_vector)) == p2(list(random_O_vector)) == 0)
    # print('random V vector \t', random_V_vector, 'evaluation \t',  p1(list(random_V_vector)) == p2(list(random_V_vector)) == 0)
    # print('random vector \t \t', random_vector, 'evaluation \t',  p1(list(random_vector)) == p2(list(random_vector)) == 0

# SET PARAMETERS
n, m, o, q = 5, 2, 2, 2

it = 10
res_vec = [0, 0, 0]
for i in range(it):
    res_vec[0] += experiment_1_sampling_vectors(n, m, o, q)[0]
    res_vec[1] += experiment_1_sampling_vectors(n, m, o, q)[1]
    res_vec[2] += experiment_1_sampling_vectors(n, m, o, q)[2]

print('oil vectors  \t \t', round(res_vec[0]/it, 2))
print('vinegar vectors \t', round(res_vec[1]/it, 2), "\t expected: \t", round(q**(-o), 2))
print('random vectors \t \t', round(res_vec[2]/it, 2), "\t expected: \t")

oil vectors  	 	 1.0
vinegar vectors 	 0.28 	 expected: 	 0.25
random vectors 	 	 0.32 	 expected: 	


In [8]:
import random 
def generate_M(n, q, p, d, order_string, prune = False, can_print = False):
    """
    input:
        q (int) characteristic of underlying field
        n (int) number of variables present
        p (list) of (QuadraticForm) a system of homogeneous quadratic equations,
        d (int) the desired degree of the output matrix
        order_string (String) the order in which the momomials should be indexed

    returns at tuple with:
        1. a macauley matrix of degree d, with (n d) columns and m(n d-2) rows
        2. a list representing the row indices (in the order specified by order_string)
        3. a list representing the columns indices (in deglex order)
    whose columns are index in lexicographical order
    """
    F = GF(q)
    R = PolynomialRing(F, n, 'x', order=order_string)
    x = R.gens()
    x_vec = vector(x)
    m = len(p)
    I = R.ideal([el**2 for el in x])
    Q = R.quotient(I)
    nb_rows_per_eq = Combinations(n, d-2).cardinality() # TODO use math library?
    
    # convert the system of equations to F/<x^2, ..., x_n^2>
    for i in range(len(p)):
        p[i] = p[i] - diagonal_matrix(p[i].diagonal())
        
    # calculate dimensions of resulting matrix
    nb_cols = Combinations(n, d).cardinality()
    nb_rows = m*nb_rows_per_eq

    # create row index
    row_index = []
    for i in range(m):
        mono = R.monomials_of_degree(d-2)
        for j in range(len(mono)):
            current_element = mono[len(mono) - j - 1]
            if current_element not in I:
                row_index.append((current_element, i))

    assert len(row_index) == nb_rows
    
    # create column index in lex order
    col_index = create_col_index(x, n, d)
    assert len(col_index) == nb_cols

    # prune the matrix
    if prune:
        is_rectangular = nb_cols > nb_rows
        if is_rectangular:
            selected_rows = random.sample(range(nb_rows), nb_cols)
            new_row_index = []
            for i in range(nb_rows):
                new_row_index[i] = row_index[selected_rows[i]] 
            row_index = new_row_index
            assert len(new_row_index) == nb_cols
        
    # fill the matrix
    M = matrix(F, len(row_index), nb_cols)
    for i in range(nb_rows):
        for j in range(nb_cols):
            poly_index = i // nb_rows_per_eq
            temp_poly = row_index[i][0]*(x_vec*p[poly_index]*x_vec)
            M[i, j] = temp_poly.monomial_coefficient(col_index[j])

    if can_print: # nb_rows can differ from len(row_index) if prune is True (idem for nb_cols)       
        print("row_index: \t", row_index)
        print('nb_rows: \t', nb_rows) 
        print('col_index: \t', len(col_index), 'nb_cols: \t', nb_cols)
        print("col_index: \t", col_index)
            
    return M, row_index, col_index

def test():
    m = 2
    n = 5
    o = 1
    q = 2
    d = 4
    p = []
    order = 'deglex'
    for i in range(m):
       p.append(generate_f(n, o, q))
    print(generate_M(n, q, p, d, order))

In [9]:
import math
import itertools


def predict_rank(n, m, o):
    v = n - o
    # TODO waarom niet gelijk?
    # s = max(0, sum(
    #     (-1)**i*binomial(m+i-1, i) * binomial(v+o, v+2*i), 
    #     "i", 0, floor(o/2) 
    # ))
    s_python = max(0, 
        sum((-1)**j * math.comb(m + j - 1, j) * math.comb(v + o, v + 2*j) 
            for j in range(o//2 + 1))
                  )
    return s_python
    
def is_admissible(n, m, o):
    v = n-o
    cond_1 = (min((o-1)/2, 2)*m > v) and (v > o)
    s = predict_rank(n, m, o)
    cond_2 = s <= 1
    return cond_1 and cond_2


def rank_via_hilbert_series(pk, q):
    P = PolynomialRing(GF(q), pk[0].nrows(), 'x', order='deglex')
    x_vec = vector(P.gens())
    
    # create ideal
    I = Ideal([x_vec*pk[i]*x_vec for i in range(len(pk))])
    
    # compute its hilbert series
    hs = I.hilbert_series()
    return hs
    
pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)

n, m, o = 5, 2, 2
rank_via_hilbert_series(pk, q)
bool_val = is_admissible(n, m, o)

In [30]:
# WEDGE ATTACK
def retrieve_kernel(n, q, pk, o, P, x_vec):
    """
    input:
    p: Sequence, a polynomial sequence
    x_vec: vector, a vector containing the generators of P

    output: 
    v: vector, kernel vector of M
    col_index: list containing (boolean) monomials of degree d, in deglex order
    M: matrix, the macaulay matrix of degree o of pk
    """
    
    # build Macaulay matrix of degree o
    M, row_index, col_index = generate_M(n, q, pk, o, 'deglex')
    p_eq_list = [x_vec*eq*x_vec for eq in pk]
    p_seq = Sequence(p_eq_list)
    # M = p_seq.macaulay_matrix(o) # uses the built-in Macaulay matrix generator
    
    # compute the kernel vector
    v = M.right_kernel()
    
    return v, col_index, M

def build_oil_space(col_index, kernel_vector, n, o, q, can_print = False):
    # assert kernel_vector.dimension() == 1
    v = n-o
    
    # calculate the basis elements corresponding to the col_index
    col_index_wedge_basis = []
    for i in range(len(col_index)):
        monomial = col_index[i]
        complement = [(1 - exp) % 2 for exp in monomial.exponents()[0]]
        # TODO check ordering of the monomials: is it not reverse?
        col_index_wedge_basis.append(complement)

    # create a list comparing the col_index with its wedge basis correspondents (for debugging purposes)
    temp = []
    for i in range(len(col_index)):
        temp.append((col_index[i],  col_index_wedge_basis[i]))
        
    # filter basis elements e_J satisfying J = {i} \union {o+1, ..., n} \ {j} for 1 <= i <= o < j <= n
    res = Matrix(GF(q), o, v)
    for k in range(len(col_index_wedge_basis)):
        current_exp = col_index_wedge_basis[k]
        # check if only one oil vector is included in the wedge)
        one_i = sum(current_exp[:o]) == 1
        # check if only one vinegar vector is not included in the wedge
        all_but_one_j = sum(current_exp[o:]) == v - 1
        if (one_i & all_but_one_j):
            # fill the matrix
            i = current_exp.index(1)
            j = current_exp[o:].index(0)
            res[i, j] = kernel_vector[1][k] # TODO here I suppose that Sage will always add the zero vector as a kernel element
            if can_print:
                print("(i, j) \t", f"({i}, {j})")
                print("o \t", o, "\t", current_exp, "one_i \t", one_i, "all_but_one_j \t", all_but_one_j)
                print(res)
    full_res = block_matrix([[identity_matrix(o), res]])
    if can_print:
            print("both_indices: \t",temp)

    return full_res

def wedge_attack(pk, n, q, o, can_print = False):
    P = PolynomialRing(GF(q), n, 'x', order='deglex')
    x_vec = vector(P.gens())
    v, column_index, M = retrieve_kernel(n, q, pk, o, P, x_vec)
    res = build_oil_space(column_index, v, n, o, q)
    if can_print:
        print(f"retrieved oil space in rref: \n{res}")
        # print("Macaulay matrix \t", M)
    return res, v, M

q = 2 
v, o, m = 4, 3, 5
n = o + v
pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
# for p in pk:
#     print(p)
#     print("----------"*5)
# print("sk \t :", sk[1])
res, v, M = wedge_attack(pk, n, q, o)

In [ ]:
def experiment_2_wedge_attack(n, m, o, q, can_print=True):
    pk, sk, P, oil, vinegar, O, V, gens_vector = generate_UOV_variables(n, m, o, q)
    x_vec = vector(P.gens())
    retrieved_space, kernel_vector, M = wedge_attack(pk, n, q, o)
    rank_kernel = kernel_vector.dimension()
    if can_print:
        # for i in range(len(pk)):
        #     print(f"vergelijking {i}: \t", x_vec*pk[i]*x_vec)
        # print("n, m, o, q: \t", n, m, o, q)
        print(f"O in reduced echelon form: \n{O.rref()}")
        assert O.rank() == o
        # print("the Macaulay matrix has rank: \t", M.rank(), f" and dimensions:\t{M.nrows()}, {M.ncols()}")
        # print("the kernel vector: \t", kernel_vector[1]) # prints only the second element. We expect the first element to be the zero matrix
        # print("the predicted rank of the kernel is: ", predict_rank(n, m, o))
        # print("the observed rank of the kernel is: ", kernel_vector.dimension())
        print(f"the retrieved oil space looks like: \t \n{retrieved_space.rref()}")
        print("--------------"*5)
    return kernel_vector.dimension(), O.rref(), retrieved_space.rref()

#######################################
# PARAMETERS                          
#######################################
q = 2
v, o, m = 4, 3, 6

n = v + o
test_it = 10
if is_admissible(n, m, o):
    counter_rank = 0
    counter_attack = 0
    for i in range(test_it):
        exp, O, retrieved = experiment_2_wedge_attack(n, m, o, q, False)
        if exp == 1:
            counter_rank += 1
        if O == retrieved:
            counter_attack += 1
        # print(f"iteratie {i}")
    print("aandeel van de testen waar de rang gelijk is aan 1: \t", round(counter_rank/test_it, 2))
    print("aandeel van de testen waar O.rref() == res.rref() is aan 1: \t", round(counter_attack/test_it, 2))
else: 
    print("parameters not admissible")
    if not min((o-1)/2, 2)*m > v:
        print("not min((o-1)/2, 2)*m > v")
    if not v > o:
        print("not v > o")
    if predict_rank(n, m, o) > 1:
        print(f"{predict_rank(n, m, o)} > 1")

In [ ]:
admissible_parameters = [
    [4, 3, 10],
    [4, 3, 5], # geeft vaak stelsels waarvan de Macaulay-matrix een dimensie groter dan 1 heeft
    [4, 3, 6], # voor m = 6 lijkt dat opgelost
    [9, 8, 5],
    [],
    [],
    [],
    [],
    [],
    []
]